# Notebook 07 – SQL Governance

### Why SQL Governance?

Healthcare data contains Protected Health Information (PHI) that must be secured before it can be accessed by different users.

This notebook implements governance controls using Unity Catalog to protect sensitive healthcare data through dynamic data masking, role-based access control, and role-specific SQL views.

These controls help ensure compliance with healthcare regulations such as HIPAA.

### Objective

The objective of this notebook is to implement SQL-based governance by creating masked views, role-specific views, and secure access policies for healthcare datasets.

### Input Tables

This notebook reads the following Gold tables:

- gold.patient_masked
- gold.patient_cohort_masked
- gold.readmission_predictions

### Output

This notebook creates:

- Dynamic masked views
- Role-specific SQL views
- Secure datasets for clinicians, analysts, and auditors



### 1. Create Dynamic Masked Views

Sensitive patient information such as names, dates of birth, addresses, phone numbers, and email addresses are dynamically masked using SQL CASE statements.

Different users see different levels of information depending on their assigned role.

In [0]:
%sql
SHOW TABLES IN db_healthcare_kl.gold;

In [0]:
%sql
SELECT *
FROM db_healthcare_kl.gold.patient_cohorts
LIMIT 5;


### 2. Create Patient Cohort Masked View

The patient cohort dataset is secured by masking sensitive demographic information while retaining attributes required for healthcare analytics and reporting.

In [0]:
%sql

CREATE OR REPLACE VIEW db_healthcare_kl.gold.patient_cohort_masked AS

SELECT

CASE
    WHEN person_id IS NOT NULL
    THEN CONCAT('****', RIGHT(person_id,4))
    ELSE NULL
END AS person_id,

cohort_name

FROM db_healthcare_kl.gold.patient_cohorts;

In [0]:
%sql

SELECT *
FROM db_healthcare_kl.gold.patient_cohort_masked
LIMIT 10;

In [0]:
%sql

CREATE OR REPLACE VIEW db_healthcare_kl.omop.person_data_masked AS

SELECT
    person_id,

    gender_concept_id,

    CONCAT(
        SUBSTRING(person_name,1,1),
        REPEAT('*', LENGTH(person_name)-1)
    ) AS person_name,

    CONCAT(
        SUBSTRING(date_of_birth,1,4),
        '-XX-XX'
    ) AS date_of_birth

FROM db_healthcare_kl.omop.person_data;

In [0]:
%sql

SELECT *
FROM db_healthcare_kl.omop.person_data_masked
LIMIT 10;

%md
### 3. Create Readmission Prediction View

Prediction results are exposed through a governed SQL view.

Clinical users can view patient-level predictions, while analysts receive only the information required for reporting and dashboard development.

In [0]:
%sql

CREATE OR REPLACE VIEW db_healthcare_kl.gold.readmission_predictions_masked AS

SELECT
    SHA2(person_id, 256) AS person_id,

    total_visits,
    max_duration_days,
    avg_duration_days,
    age,
    gender_concept_id,
    diagnosis_count,
    medication_count,
    previous_admissions,
    visit_type,

    CONCAT(SUBSTRING(primary_diagnosis,1,3),'***') AS primary_diagnosis,

    Cohort_name,

    SHA2(visit_occurrence_id,256) AS visit_occurrence_id,

    visit_end_date,
    next_visit_start_date,

    readmission_30day_flag,

    processed_timestamp,

    prediction

FROM db_healthcare_kl.gold.readmission_predictions;

In [0]:
%sql

SELECT *
FROM db_healthcare_kl.gold.readmission_predictions_masked
LIMIT 10;